In [1]:
from transformers import pipeline

# Load an Arabic sentiment model trained on Arabic dialects (CAMeL Lab, NYU Abu Dhabi)
sentiment = pipeline("sentiment-analysis",
                     model="CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment")

# Test it on a few examples before running on all reviews
examples = [
    "التطبيق ممتاز والتوصيل سريع",
    "اسوء تطبيق والمندوب تاخر ساعتين",
    "عادي",
    "حلو بس الاسعار غاليه شوي",
]

for text, result in zip(examples, sentiment(examples)):
    print(result["label"], round(result["score"], 2), "|", text)

c:\Users\saleh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\saleh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\saleh\.cache\huggingface\hub\models--CAMeL-Lab--bert-base-arabic-camelbert-da-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mo

positive 0.68 | التطبيق ممتاز والتوصيل سريع
negative 0.99 | اسوء تطبيق والمندوب تاخر ساعتين
negative 0.7 | عادي
positive 0.42 | حلو بس الاسعار غاليه شوي


In [2]:
import pandas as pd
import time

df = pd.read_csv("data/clean/reviews_clean.csv", parse_dates=["at"])
sample = df.sample(1000, random_state=42).copy()

# Run the AI model on 1,000 reviews and measure the time
start = time.time()
preds = sentiment(sample["content"].astype(str).tolist(), batch_size=32)
elapsed = time.time() - start

sample["ai_label"] = [p["label"] for p in preds]
sample["ai_score"] = [p["score"] for p in preds]

print(f"Time for 1,000 reviews: {elapsed:.0f} seconds")
print(f"Estimated time for all {len(df):,} reviews: {elapsed * len(df) / 1000 / 60:.0f} minutes")
print("-" * 40)

# Compare AI judgment with the stars (% of each star level)
(pd.crosstab(sample["score"], sample["ai_label"], normalize="index") * 100).round(1)

Time for 1,000 reviews: 105 seconds
Estimated time for all 46,025 reviews: 81 minutes
----------------------------------------


ai_label,negative,neutral,positive
score,,,
1,78.5,16.7,4.8
2,66.7,26.7,6.7
3,40.9,45.5,13.6
4,31.0,24.1,44.8
5,3.9,10.0,86.1


In [3]:
import os

os.makedirs("data/ai", exist_ok=True)
chunk_size = 2000

for i in range(0, len(df), chunk_size):
    path = f"data/ai/chunk_{i}.csv"
    if os.path.exists(path):
        continue  # this chunk is already done (useful if we stop and restart)

    chunk = df.iloc[i:i + chunk_size][["reviewId", "content"]].copy()
    preds = sentiment(chunk["content"].astype(str).tolist(), batch_size=32)
    chunk["ai_label"] = [p["label"] for p in preds]
    chunk["ai_score"] = [round(p["score"], 3) for p in preds]
    chunk[["reviewId", "ai_label", "ai_score"]].to_csv(path, index=False)

    print(f"Done {min(i + chunk_size, len(df)):,} / {len(df):,}")

print("ALL DONE!")

Done 2,000 / 46,025
Done 4,000 / 46,025
Done 6,000 / 46,025
Done 8,000 / 46,025
Done 10,000 / 46,025
Done 12,000 / 46,025
Done 14,000 / 46,025
Done 16,000 / 46,025
Done 18,000 / 46,025
Done 20,000 / 46,025
Done 22,000 / 46,025
Done 24,000 / 46,025
Done 26,000 / 46,025
Done 28,000 / 46,025
Done 30,000 / 46,025
Done 32,000 / 46,025
Done 34,000 / 46,025
Done 36,000 / 46,025
Done 38,000 / 46,025
Done 40,000 / 46,025
Done 42,000 / 46,025
Done 44,000 / 46,025
Done 46,000 / 46,025
Done 46,025 / 46,025
ALL DONE!


In [4]:
import glob

# Combine all chunk files into one table
ai = pd.concat([pd.read_csv(f) for f in glob.glob("data/ai/chunk_*.csv")])
print("AI results:", len(ai))

# Join AI results to the main data using reviewId
df = df.drop(columns=["ai_label", "ai_score"], errors="ignore")
df = df.merge(ai, on="reviewId", how="left")
print("Reviews without AI label:", df["ai_label"].isna().sum())

# Save
df.to_csv("data/clean/reviews_clean.csv", index=False, encoding="utf-8-sig")
print("Saved!", df.shape)
print("-" * 40)

# Compare AI with stars on ALL reviews
(pd.crosstab(df["score"], df["ai_label"], normalize="index") * 100).round(1)

AI results: 46025
Reviews without AI label: 0
Saved! (46025, 24)
----------------------------------------


ai_label,negative,neutral,positive
score,,,
1,78.7,17.0,4.3
2,58.6,30.6,10.8
3,39.3,34.0,26.8
4,20.8,26.3,52.9
5,4.3,8.5,87.2


In [5]:
pd.set_option("display.max_colwidth", 150)

confident = df["ai_score"] >= 0.9
five_star_negative = df[(df["score"] == 5) & (df["ai_label"] == "negative") & confident]
one_star_positive  = df[(df["score"] == 1) & (df["ai_label"] == "positive") & confident]

print("5 stars but negative text:", len(five_star_negative))
print("1 star but positive text: ", len(one_star_positive))

print("\n===== 5 STARS but NEGATIVE text =====")
display(five_star_negative[["app", "content"]].sample(min(10, len(five_star_negative)), random_state=1))

print("\n===== 1 STAR but POSITIVE text =====")
display(one_star_positive[["app", "content"]].sample(min(10, len(one_star_positive)), random_state=1))

5 stars but negative text: 315
1 star but positive text:  374

===== 5 STARS but NEGATIVE text =====


,app,content
27173,Keeta,حلو لاكن ليته بدون هويه مراااااااااااااااااااا حلو بس ليته مافيه هويه رقم هويه مراااااااااااااااااااا قهرني ولا التطبيق خياللللللللللللللللللللل
26587,Keeta,التطبيق رائع الكثير من الزبائن يطلبون تونه بالحجر او شكشوكه بالحجر او عريكه بالحجر هم يخسرون انفسهم على الفاضي لانه هل مطعم سيعطيك حجر سفري لا يمك...
39714,Mrsool,حسبنا الله ونعم الوكيل في كل واحد قعد المناديب الناس بقالهم سنين شغللة معاكم وانتو بقرار غبى مش بقين على الناس الاى طلعتكوا منكم لله فى الشهر المف...
31120,Keeta,افضل مليون مرة من غيره من المنافشين وبالذات هذولاك اللي يقولك اشترك ونعطيك توصيل مجاني وفي النهاية انت اصلا دافعه ولا وفرت شي ويرفعو اسعار المنتجا...
29499,Keeta,بس ماعد في تخفيضات
9024,HungerStation,برنمج فطيع
14457,Jahez,لابئس
27443,Keeta,احسن من مرسول و هنقرستيشن رخيص كمان
12543,HungerStation,التحديث الأخير مرة سئ
42311,Mrsool,للاسف صار البرنامج مو زين اسعاره غير مناسبة



===== 1 STAR but POSITIVE text =====


,app,content
8207,HungerStation,نجمة وكثيرة عليكم
39630,Mrsool,كان ممتاز. زمان وانتهى
34399,Mrsool,كثير عليه نجمه تحياتي لكيتا عروضه احسن
33331,Mrsool,تجاري المتعددة مع مرسول كلها ممتازة و انا ممتنة لهم و أحب التعامل معهم ...شكرا مرسول .
10460,HungerStation,اسعاركم ناررر غاليين مره
45585,ToYou,ماشاء الله كل شي مزودين فيه خمسه ريال وهاذا غير صلخ التوصيل احسنتم الى الامام
18041,Jahez,السعودي وشعبها خط احمر
39860,Mrsool,جيد
8052,HungerStation,اسعار نااااار
16783,Jahez,يطول الطلب👎


In [6]:
topic_cols = ["Customer service", "Price", "Driver", "Payment & refund", "Delay",
              "Order issue", "Offers", "Fraud accusations", "App technical", "Cancellation"]

# Hidden complaint = 4-5 stars, but the AI is confident the text is negative
df["hidden_complaint"] = (df["score"] >= 4) & (df["ai_label"] == "negative") & (df["ai_score"] >= 0.9)

high = df[df["score"] >= 4]
print("Hidden complaint rate (% of 4-5 star reviews):")
print((high.groupby("app")["hidden_complaint"].mean() * 100).round(1).sort_values(ascending=False))
print("-" * 40)

# What do these 'happy' customers still complain about?
hidden = df[df["hidden_complaint"]]
print("Hidden complaints:", len(hidden))
(hidden[topic_cols].mean() * 100).round(1).sort_values(ascending=False)

Hidden complaint rate (% of 4-5 star reviews):
app
Jahez            2.6
ToYou            2.2
HungerStation    2.1
Mrsool           2.1
Keeta            1.8
Name: hidden_complaint, dtype: float64
----------------------------------------
Hidden complaints: 436


Price                16.1
Driver               15.4
Payment & refund      9.6
Offers                9.4
Delay                 7.8
Customer service      7.6
App technical         5.5
Fraud accusations     5.3
Order issue           5.0
Cancellation          0.9
dtype: float64

In [7]:
import pyodbc
import urllib
from sqlalchemy import create_engine
from sqlalchemy.types import NVARCHAR

# Save the CSV with the new AI columns
df["repliedAt"] = pd.to_datetime(df["repliedAt"])
df.to_csv("data/clean/reviews_clean.csv", index=False, encoding="utf-8-sig")

# Connect to SQL Server
driver = [d for d in pyodbc.drivers() if "SQL Server" in d][-1]
params = urllib.parse.quote_plus(
    f"DRIVER={{{driver}}};SERVER=SALEH\\SQLEXPRESS11;DATABASE=DeliveryReviews;"
    "Trusted_Connection=yes;TrustServerCertificate=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}", fast_executemany=True)

text_types = {
    "reviewId": NVARCHAR(100),
    "content": NVARCHAR(1000),
    "clean_text": NVARCHAR(1000),
    "replyContent": NVARCHAR(2000),
    "reviewCreatedVersion": NVARCHAR(50),
    "appVersion": NVARCHAR(50),
    "app": NVARCHAR(50),
    "topics_list": NVARCHAR(300),
    "ai_label": NVARCHAR(20),
}

df.to_sql("reviews", engine, if_exists="replace", index=False,
          dtype=text_types, chunksize=1000)
print("SQL Server updated!", df.shape)

SQL Server updated! (46025, 25)
